In [ ]:
options(encoding = "UTF-8")
Sys.setlocale("LC_ALL", "pl_PL.UTF-8")

# Sprawdzenie i instalacja brakujących pakietów
required_packages <- c(
  "tidyverse", "broom", "car", "performance", 
  "corrplot", "lmtest", "olsrr", "caret", "recipes"
)

new_packages <- required_packages[!(required_packages %in% installed.packages()[,"Package"])]
if(length(new_packages)) install.packages(new_packages)

library(tidyverse)
library(broom)
library(car)
library(performance)
library(corrplot)
library(lmtest)
library(olsrr)
library(caret)
library(recipes)

seed <- 1333
set.seed(seed)

Funkcje pomocnicze 

In [ ]:
vifs_calculation <- function(df, entry_name = NULL) {
  temp_df <- cbind(df, y_temp = rnorm(nrow(df)))
  vif_results <- car::vif(lm(y_temp ~ ., data = temp_df))
  
  if (!is.null(entry_name)) {
    vif_df <- data.frame(NAME_INDEX = entry_name, t(vif_results)) %>% 
      tibble::column_to_rownames("NAME_INDEX")
  } else {
    vif_df <- data.frame(t(vif_results))
  }
  return(vif_df)
}

F_test_permutation <- function(df, y, test_cols, n_perm) {
  results <- list()
  X_reduced <- df %>% select(-all_of(test_cols))
  
  for (test_col in test_cols) {
    X_full <- df %>% select(-all_of(setdiff(test_cols, test_col)))
    
    model_reduced <- lm(y ~ ., data = cbind(X_reduced, y = y))
    model_full <- lm(y ~ ., data = cbind(X_full, y = y))
    
    f_stat <- anova(model_reduced, model_full)$F[2]
    
    f_stats <- replicate(n_perm, {
      X_perm <- X_full
      X_perm[[test_col]] <- sample(X_full[[test_col]])
      anova(model_reduced, lm(y ~ ., data = cbind(X_perm, y = y)))$F[2]
    })
    
    p_value <- (sum(f_stats >= f_stat) + 1) / (n_perm + 1)
    
    results[[test_col]] <- data.frame(
      `Testowana zmienna` = test_col,
      `Statystyka F` = f_stat,
      `P value` = p_value,
      check.names = FALSE
    )
  }
  bind_rows(results) %>% tibble::column_to_rownames("Testowana zmienna")
}

Ładowanie danych

In [ ]:
COL_NAMES <- c('Płeć', 'Długość', 'Średnica', 'Wysokość', 'Cała_waga', 
               'Waga_po_obraniu', 'Waga_trzewi', 'Waga_powłoki', 'Pierścienie')

if (!file.exists('abalone_data_set.csv')) {
  stop("Brak pliku danych. Proszę zapewnić 'abalone_data_set.csv'")
} else {
  df <- read.csv('abalone_data_set.csv', col.names = COL_NAMES, fileEncoding = "UTF-8")
}

X_reduced <- df %>% select(-Pierścienie)
y <- df %>% select(Pierścienie)

X_reduced <- X_reduced %>% 
  mutate(
    Płeć_M = as.numeric(Płeć == "M"),
    Płeć_F = as.numeric(Płeć == "F"),
    Płeć_I = as.numeric(Płeć == "I")
  ) %>% 
  select(Płeć_I, Płeć_M, Płeć_F, everything(), -Płeć)

In [ ]:
SEX <- 'I'
filt <- X_reduced[[paste0('Płeć_', SEX)]] == 1
sex_columns <- c('Płeć_I', 'Płeć_M', 'Płeć_F')

y_local <- y[filt, , drop = FALSE]
X_old <- X_reduced[filt, !(names(X_reduced) %in% sex_columns), drop = FALSE]

X_new <- X_old %>% 
  mutate(
    `Długość/Wysokość` = Długość / Wysokość,
    `Masa/objętość` = Cała_waga / (Długość * Średnica * Wysokość),
    `Waga_powłoki/Cała_waga` = Waga_powłoki / Cała_waga,
    `Waga_po_obraniu/Cała_waga` = Waga_po_obraniu / Cała_waga,
    `Waga_trzewi/Cała_waga` = Waga_trzewi / Cała_waga,
    `Waga_powłoki/Waga_po_obraniu` = Waga_powłoki / Waga_po_obraniu,
    `Waga_trzewi/Waga_po_obraniu` = Waga_trzewi / Waga_po_obraniu,
    `Waga_trzewi/Waga_powłoki` = Waga_trzewi / Waga_powłoki
  )

invalid_rows <- rowSums(is.na(X_new) | is.infinite(as.matrix(X_new))) > 0
X_old <- X_old[!invalid_rows, , drop = FALSE]
X_new <- X_new[!invalid_rows, , drop = FALSE] 
y_local <- y_local[!invalid_rows, , drop = FALSE]

NEW_FEATURE_NAMES <- setdiff(names(X_new), names(X_old))

Analiza modelu dla konkretnej płci 

In [ ]:
model_data <- cbind(X_old, Pierścienie = y_local$Pierścienie)
model_old <- lm(Pierścienie ~ ., data = model_data)

# Diagnostyka
residuals <- residuals(model_old)

par(mfrow = c(1,2))
plot(fitted(model_old), residuals, main = "Reszty vs dopasowane")
abline(h = 0, col = "red")
qqnorm(residuals, main = "Wykres QQ")
qqline(residuals, col = "red")

cat("Test Shapiro-Wilka p-value:", shapiro.test(residuals)$p.value, "\n")
cat("Test Breusch-Pagan p-value:", bptest(model_old)$p.value, "\n")

Normalność modelu

Redukcja nowych zmiennych 

In [ ]:
# VIF dla modelu podstawowego
reduce_basic <- c('Cała_waga', 'Długość')
vifs_basic <- vifs_calculation(X_old, "Model bazowy")
print(vifs_basic)

# VIF po redukcji podstawowych zmiennych
vifs_basic_after_red <- vifs_calculation(dplyr::select(X_old, -all_of(reduce_basic)), "Model bazowy po redukcji")
print(vifs_basic_after_red)

# Tworzenie ostatecznego zbioru cech
X_final <- dplyr::select(X_new, -all_of(reduce_basic))

# Porównanie R-squared
model_new <- lm(Pierścienie ~ ., data = cbind(X_old, Pierścienie = y_local$Pierścienie))
model_final <- lm(Pierścienie ~ ., data = cbind(dplyr::select(X_old, -all_of(reduce_basic)), Pierścienie = y_local$Pierścienie))

cat("R-squared początkowy model:", summary(model_new)$r.squared, "\n")
cat("R-squared model po redukcji:", summary(model_final)$r.squared, "\n")

# Test F z permutacjami
n_perm <- 1000  # Liczba permutacji
F_test_before_reduction <- F_test_permutation(X_final, y_local$Pierścienie, NEW_FEATURE_NAMES, n_perm)
print(F_test_before_reduction)

# KOŃCOWA ANALIZA - VIFY PO REDUKCJI
significant_level <- 0.05
vif_reduce_value <- 10

F_test_before_reduction$`Odrzucenie H0` <- F_test_before_reduction$`P value` < significant_level
reduce_features_F_test <- rownames(F_test_before_reduction)[!F_test_before_reduction$`Odrzucenie H0`]

# Redukcja modelu na podstawie VIF
reduce_features_vif_test <- c()
vifs_after_F_test <- list()
index_name <- "Model rozszerzony przed redukcją vif"

repeat {
  drop_features <- c(reduce_features_F_test, reduce_features_vif_test)
  current_features <- setdiff(colnames(X_final), drop_features)
  local_vifs <- vifs_calculation(dplyr::select(X_final, all_of(current_features)), index_name)
  
  max_vif_column <- names(which.max(local_vifs[1, ]))
  max_vif_value <- max(local_vifs[1, ])
  
  vifs_after_F_test[[length(vifs_after_F_test) + 1]] <- local_vifs
  reduce_features_vif_test <- c(reduce_features_vif_test, max_vif_column)
  
  if (max_vif_value < vif_reduce_value) break
}

vifs_after_F_test <- dplyr::bind_rows(vifs_after_F_test)

# Porównanie modeli końcowych
model_new <- lm(Pierścienie ~ ., data = cbind(X_new, Pierścienie = y_local$Pierścienie))
model_final <- lm(Pierścienie ~ ., data = cbind(X_final, Pierścienie = y_local$Pierścienie))

cat("R-squared model pełny:", summary(model_new)$r.squared, "\n")
cat("R-squared model końcowy:", summary(model_final)$r.squared, "\n")

In [ ]:
# -------- TEST F Z PERMUTACJAMI I REDUKCJA MODELU -------- #

# Ustawienie poziomu istotności i progu VIF (zmienione z 10 na 12)
significant_level <- 0.05
vif_reduce_value <- 12

# Dodanie kolumny informującej o odrzuceniu H0 przy danym poziomie istotności
col_name_h0 <- paste0("Odrzucenie H0 dla alpha = ", significant_level)
F_test_before_reduction[[col_name_h0]] <- F_test_before_reduction$`P value` < significant_level

# Wybór zmiennych, dla których nie odrzucono H0 (czyli P value >= 0.05)
reduce_features_F_test <- rownames(
  F_test_before_reduction[
    !F_test_before_reduction[[col_name_h0]], 
  ]
)

# Iteracyjna redukcja na podstawie VIF (zaktualizowana do progu 12)
reduce_features_vif_test <- c()
vifs_after_F_test <- list()
index_name <- "Model rozszerzony przed redukcją vif"

repeat {
  drop_features <- c(reduce_features_F_test, reduce_features_vif_test)
  current_features <- setdiff(colnames(X_final), drop_features)
  local_vifs <- vifs_calculation(
    dplyr::select(X_final, all_of(current_features)), 
    index_name
  )
  
  max_vif_column <- names(which.max(local_vifs[1, ]))
  max_vif_value  <- max(local_vifs[1, ])
  
  vifs_after_F_test[[length(vifs_after_F_test) + 1]] <- local_vifs
  reduce_features_vif_test <- c(reduce_features_vif_test, max_vif_column)
  
  if (max_vif_value < vif_reduce_value) break
}

vifs_after_F_test <- dplyr::bind_rows(vifs_after_F_test)

# Ostateczne modele i porównanie R-squared
model_new   <- lm(Pierścienie ~ ., data = cbind(X_new,   Pierścienie = y_local$Pierścienie))
model_final <- lm(Pierścienie ~ ., data = cbind(X_final, Pierścienie = y_local$Pierścienie))

cat("R-squared model pełny:   ", summary(model_new)$r.squared, "\n")
cat("R-squared model końcowy: ", summary(model_final)$r.squared, "\n")


In [ ]:
# -------- DODATKOWA ANALIZA: REGRESJA Z BOX–COX I USUWANIE OUTLIERÓW -------- #

library(MASS)     # potrzebne do funkcji boxcox()
library(car)      # potrzebne do odległości Cooka
library(tibble)   # do ładnego łączenia wyników

set.seed(seed)

# 1) Podział na zbiór uczący i testowy (75% / 25%)
n_total   <- nrow(X_final)
train_idx <- sample(seq_len(n_total), size = floor(0.75 * n_total))
test_idx  <- setdiff(seq_len(n_total), train_idx)

train_data <- cbind(Pierścienie = y_local$Pierścienie[train_idx], X_final[train_idx, , drop = FALSE])
test_data  <- cbind(Pierścienie = y_local$Pierścienie[test_idx],  X_final[test_idx,  , drop = FALSE])

# Przygotowanie ramki na metryki
metric_names <- c(
  "Probe_size", 
  "R2_train", 
  "R2_test", 
  "Lambda_hat", 
  "LR_p", 
  "Shapiro_train_p", 
  "Shapiro_train_bc_p", 
  "Shapiro_test_p", 
  "Shapiro_test_bc_p"
)
metrics_list <- vector("list", 7)
names(metrics_list) <- paste0("iter_", 0:6)

# Indeksy obserwacji w bieżącym zbiorze uczącym (początkowo wszystkie)
current_idx <- seq_len(nrow(train_data))

for (i in seq_len(7)) {
  # 2) Wyodrębnienie podzbioru uczącego bez usuniętych odchyleń
  train_loc <- train_data[current_idx, , drop = FALSE]
  y_loc     <- train_loc$Pierścienie
  
  # 3) Dopasowanie modelu bez transformacji
  model_raw <- lm(Pierścienie ~ ., data = train_loc)
  
  # 4) Obliczenie R^2 dla zbioru uczącego i testowego
  r2_train <- summary(model_raw)$r.squared
  preds_test <- predict(model_raw, newdata = test_data)
  ss_res <- sum((test_data$Pierścienie - preds_test)^2)
  ss_tot <- sum((test_data$Pierścienie - mean(test_data$Pierścienie))^2)
  r2_test <- 1 - ss_res / ss_tot
  
  # 5) Reszty surowe i test Shapiro–Wilka (tylko jeśli co najmniej 3 obserwacje)
  resid_train <- resid(model_raw)
  resid_test  <- test_data$Pierścienie - preds_test
  
  sh_tr_p <- if (length(resid_train) >= 3) shapiro.test(resid_train)$p.value else NA
  sh_te_p <- if (length(resid_test)  >= 3) shapiro.test(resid_test)$p.value  else NA
  
  # 6) Estymacja lambda przez MLE (funkcja boxcox z argumentem plotit = FALSE zwraca profile log-lik)
  bc_raw <- boxcox(Pierścienie ~ ., data = train_loc, plotit = FALSE)
  lambda_seq   <- bc_raw$x
  loglik_seq   <- bc_raw$y
  lambda_hat   <- lambda_seq[which.max(loglik_seq)]
  
  # 7) Test ilorazu wiarygodności: porównanie z lambda = 1 (brak transformacji)
  loglik_hat    <- max(loglik_seq)
  idx_lambda1   <- which.min(abs(lambda_seq - 1))
  loglik_lambda1 <- loglik_seq[idx_lambda1]
  
  lr_stat <- 2 * (loglik_hat - loglik_lambda1)
  lr_p    <- 1 - pchisq(lr_stat, df = 1)
  
  # 8) Transformacja y według Box–Cox i dopasowanie modelu na y^*
  train_loc$Pierścienie_bc <- if (lambda_hat == 0) {
    log(train_loc$Pierścienie)
  } else {
    (train_loc$Pierścienie^lambda_hat - 1) / lambda_hat
  }
  test_y_bc <- if (lambda_hat == 0) {
    log(test_data$Pierścienie)
  } else {
    (test_data$Pierścienie^lambda_hat - 1) / lambda_hat
  }
  
  model_bc <- lm(Pierścienie_bc ~ ., data = train_loc %>% select(-Pierścienie))
  
  # 9) Reszty po transformacji i test Shapiro–Wilka
  resid_train_bc <- resid(model_bc)
  resid_test_bc  <- test_y_bc - predict(model_bc, newdata = test_data %>% select(-Pierścienie))
  
  sh_tr_bc_p <- if (length(resid_train_bc) >= 3) shapiro.test(resid_train_bc)$p.value else NA
  sh_te_bc_p <- if (length(resid_test_bc)  >= 3) shapiro.test(resid_test_bc)$p.value  else NA
  
  # 10) Zapisanie wyników iteracji
  metrics_list[[paste0("iter_", i - 1)]] <- c(
    Probe_size        = nrow(train_loc),
    R2_train          = round(r2_train, 6),
    R2_test           = round(r2_test,  6),
    Lambda_hat        = round(lambda_hat, 6),
    LR_p              = round(lr_p, 6),
    Shapiro_train_p   = round(sh_tr_p, 6),
    Shapiro_train_bc_p= round(sh_tr_bc_p, 6),
    Shapiro_test_p    = round(sh_te_p, 6),
    Shapiro_test_bc_p = round(sh_te_bc_p, 6)
  )
  
  # 11) Jeżeli nie jest to ostatnia iteracja, usuń obserwacje odstające według Cooka
  if (i < 7) {
    cooks_d <- cooks.distance(model_raw)
    thresh  <- 4 / nrow(train_loc)
    # Zachowaj tylko te obserwacje, dla których Cook’s D < prog
    keep_idx <- which(cooks_d < thresh)
    current_idx <- current_idx[keep_idx]
  }
}

# 12) Utworzenie ramki wyników i zapis do pliku
results_df <- as_tibble(do.call(cbind, metrics_list), rownames = "metric")
results_df <- column_to_rownames(results_df, var = "metric")

# Zakładamy istnienie funkcji save_file_safe()
save_file_safe(results_df, 'boxcox_leverage', 'csv', df_index = TRUE, skip_if_exists = TRUE)

# 13) Wyświetlenie rezultatu w konsoli
print(results_df)
